# บทที่ 9: การสำรวจและตรวจสอบคุณภาพข้อมูลเบื้องต้น

หลังจากนำเข้าข้อมูลแล้ว ขั้นตอนถัดไปคือการสำรวจว่า DataFrame ที่ได้มีโครงสร้างและคุณภาพเพียงพอสำหรับการวิเคราะห์หรือไม่

กระบวนการนี้เรียกว่า **Exploratory Data Analysis (EDA)** หรือการสำรวจข้อมูลเบื้องต้น

จุดสำคัญของบทนี้คือ

> การอ่านข้อมูลสำเร็จไม่ได้หมายความว่าข้อมูลพร้อมสำหรับการวิเคราะห์ เราต้องตรวจสอบโครงสร้าง ชนิดข้อมูล ค่าว่าง ข้อมูลซ้ำ ช่วงค่า และความสอดคล้องของข้อมูลก่อนเสมอ

ผลจากการตรวจสอบในบทนี้จะถูกนำไปใช้วางแผนการทำความสะอาดข้อมูลในบทถัดไป

## ผลการเรียนรู้ที่คาดหวัง

เมื่อเรียนจบบทนี้ ผู้เรียนจะสามารถ

1. ตรวจสอบโครงสร้างและชนิดข้อมูลของ DataFrame ได้
2. สรุปและประเมินข้อมูลตัวเลขและข้อมูลเชิงหมวดหมู่ได้
3. ตรวจหาค่าว่าง ข้อมูลซ้ำ และค่าที่อยู่นอกช่วงได้
4. สร้างตารางสรุปประเด็นคุณภาพข้อมูลเพื่อส่งต่อไปยังขั้นตอน Data Cleaning ได้

## ลำดับเนื้อหา

บทเรียนนี้ประกอบด้วยหัวข้อต่อไปนี้

1. เตรียม pandas และข้อมูลตัวอย่าง
2. แนวคิด EDA และการตรวจสอบคุณภาพข้อมูล
3. ดูตัวอย่างข้อมูล
4. ตรวจสอบขนาด ชื่อคอลัมน์ และ index
5. ตรวจสอบชนิดข้อมูลและภาพรวม DataFrame
6. ทำความเข้าใจค่าสถิติพื้นฐาน
7. สรุปข้อมูลตัวเลขด้วย `describe()`
8. ตรวจสอบค่าที่อยู่นอกช่วง
9. ตรวจสอบค่าว่าง
10. ตรวจสอบข้อมูลซ้ำ
11. ตรวจสอบข้อมูลเชิงหมวดหมู่
12. ตรวจสอบข้อมูลวันที่
13. สร้างตารางสรุปคุณภาพข้อมูล
14. แบบฝึกหัดท้ายบท

## 1. เตรียม pandas และข้อมูลตัวอย่าง

เริ่มจาก import pandas และ `Path`

ในบทนี้จะใช้ไฟล์

```python
"msdhs_ops_mso_logbook.csv"
```

In [1]:
from pathlib import Path

import pandas as pd

In [2]:
csv_file = Path(
    "msdhs_ops_mso_logbook.csv"
)

csv_file

PosixPath('msdhs_ops_mso_logbook.csv')

ก่อนอ่านข้อมูล ควรตรวจสอบว่าไฟล์มีอยู่จริงและ Path นั้นเป็นไฟล์

In [3]:
print("File exists:", csv_file.exists())
print("Is a file:", csv_file.is_file())

File exists: True
Is a file: True


เลือกเฉพาะคอลัมน์ที่ใช้ในการฝึกตรวจสอบคุณภาพข้อมูล เช่น

- วันที่
- รหัส
- จังหวัด
- อายุ
- เพศ
- ระดับการศึกษา
- อาชีพ
- ประเภทกลุ่มเป้าหมาย

In [4]:
mso_usecols = [
    "วันที่แก้ไขข้อมูลล่าสุด",
    "รหัสครัวเรือน",
    "รหัสประจำบ้าน",
    "วันที่สร้างครัวเรือน",
    "จังหวัด",
    "อายุ",
    "เพศ",
    "ระดับการศึกษา",
    "อาชีพหลัก",
    "ประเภทกลุ่มเป้าหมาย",
    "รหัส cm",
    "หน่วยงานของ cm",
]

mso_dtype = {
    "รหัสครัวเรือน": "string",
    "รหัสประจำบ้าน": "string",
    "รหัส cm": "string",
}

In [5]:
mso_df = pd.read_csv(
    csv_file,
    encoding="utf-8-sig",
    usecols=mso_usecols,
    dtype=mso_dtype,
    parse_dates=[
        "วันที่แก้ไขข้อมูลล่าสุด",
        "วันที่สร้างครัวเรือน",
    ],
)

mso_df


,วันที่แก้ไขข้อมูลล่าสุด,รหัสครัวเรือน,รหัสประจำบ้าน,วันที่สร้างครัวเรือน,อายุ,เพศ,จังหวัด,ระดับการศึกษา,อาชีพหลัก,ประเภทกลุ่มเป้าหมาย,รหัส cm,หน่วยงานของ cm
0,2023-06-19 13:40:56.585,648ff7d6b64d06b6b0dc7f02,<NA>,2023-06-19 13:38:14.881,3,หญิง,อุดรธานี,ไม่ได้เรียนหนังสือ,เกษตรกรรม (พืช ปศุสัตว์ ประมง),เด็กเล็ก,cm410011,สำนักงานพัฒนาสังคมและความมั่นคงของมนุษย์จังหวั...
1,2022-07-17 18:56:10.147,62d3f7a6d6f101550054198b,70040223124,2022-07-17 18:51:02.889,55,หญิง,ราชบุรี,ไม่ได้เรียนหนังสือ,NaN,วัยผู้ใหญ่/วัยแรงงาน,cm700018,ศูนย์คุ้มครองคนไร้ที่พึ่งราชบุรี
2,2024-01-25 13:34:46.941,62a05c9df5674ecdd3805787,<NA>,2022-08-06 15:23:57.850,33,หญิง,หนองคาย,มัธยมศึกษาตอนต้น,NaN,วัยผู้ใหญ่/วัยแรงงาน,cm430010,สำนักงานพัฒนาสังคมและความมั่นคงของมนุษย์จังหวั...
3,2024-03-22 14:12:39.290,62908e378fa67bf5ad7d5555,<NA>,2022-05-27 15:39:19.197,5,หญิง,อุบลราชธานี,ไม่ได้เรียนหนังสือ,NaN,เด็กเล็ก,cm340009,สำนักงานพัฒนาสังคมและความมั่นคงของมนุษย์จังหวั...
4,2023-03-15 09:27:11.763,6319b26ca3a37508384e36fc,<NA>,2022-08-09 16:14:20.652,3,ชาย,นครราชสีมา,ไม่ได้เรียนหนังสือ,NaN,เด็กเล็ก,cm300020,ศูนย์บริการคนพิการ จังหวัดนครราชสีมา
...,...,...,...,...,...,...,...,...,...,...,...,...
95,2025-07-29 14:33:32.178,688878eaa933a405b21cc136,<NA>,2025-07-29 14:31:54.610,59,NaN,ชัยภูมิ,NaN,เกษตรกรรม (พืช ปศุสัตว์ ประมง),วัยผู้ใหญ่/วัยแรงงาน,cm360060,ศูนย์คุ้มครองคนไร้ที่พึ่ง จ.ชัยภูมิ
96,2023-03-17 19:08:15.489,628b008c8fa67bf5ad7d221b,<NA>,2022-05-23 10:33:32.270,79,NaN,มหาสารคาม,NaN,NaN,ผู้สูงอายุ,cm440005,สำนักงานพัฒนาสังคมและความมั่นคงของมนุษย์จังหวั...
97,2023-05-01 09:19:26.395,63b6325b2617b2b7aa04dc0b,25120251099,2023-05-01 09:13:47.520,51,หญิง,สระแก้ว,ประถมศึกษา,รับจ้างทั่วไป,วัยผู้ใหญ่/วัยแรงงาน,cm270001,สำนักงานพัฒนาสังคมและความมั่นคงของมนุษย์จังหวั...
98,2025-01-04 14:05:25.955,63e1d0d92617b2b7aa06096d,40160104483,2023-07-02 11:17:29.369,81,ชาย,ขอนแก่น,ไม่ได้เรียนหนังสือ,NaN,ผู้สูงอายุ,cm400016,ศูนย์เรียนรู้การพัฒนาสตรีและครอบครัวรัตนาภา จั...


## 2. แนวคิด EDA และการตรวจสอบคุณภาพข้อมูล

EDA ย่อมาจาก **Exploratory Data Analysis**

หมายถึงการสำรวจข้อมูลเบื้องต้นเพื่อทำความเข้าใจว่า

- DataFrame มีหน้าตาอย่างไร
- ข้อมูลมีขนาดเท่าไร
- แต่ละคอลัมน์หมายถึงอะไร
- คอลัมน์ถูกอ่านเป็นชนิดข้อมูลใด
- มีค่าว่างหรือข้อมูลซ้ำหรือไม่
- ช่วงของข้อมูลสมเหตุสมผลหรือไม่
- ค่าหมวดหมู่มีความสอดคล้องกันหรือไม่
- ข้อมูลพร้อมสำหรับการวิเคราะห์หรือยัง

ในขั้นตอน EDA เรามุ่งเน้นที่การ **ตรวจพบและบันทึกปัญหา** ก่อนตัดสินใจแก้ไขข้อมูล

ตัวอย่างปัญหาที่อาจพบหลังนำเข้าข้อมูล ได้แก่

- pandas อ่านชนิดข้อมูลไม่ตรงกับความหมายจริง
- วันที่ยังเป็นข้อความ
- รหัสถูกอ่านเป็นตัวเลข
- มีค่าว่างในคอลัมน์สำคัญ
- มีข้อมูลซ้ำ
- อายุหรือค่าตัวเลขอยู่นอกช่วงที่สมเหตุสมผล
- หมวดหมู่เดียวกันสะกดหลายรูปแบบ
- วันที่อยู่ก่อนหรือหลังช่วงเวลาที่คาดหวัง
- บางคอลัมน์มีค่าว่างมากจนไม่เหมาะกับการใช้งาน

In [2]:
csv_file = "msdhs_ops_mso_logbook.csv"

## 3. ดูตัวอย่างข้อมูล

การดูตัวอย่างข้อมูลช่วยตรวจสอบอย่างรวดเร็วว่า

- ข้อมูลถูกอ่านเป็นตารางหรือไม่
- ชื่อคอลัมน์ถูกต้องหรือไม่
- ภาษาไทยแสดงผลปกติหรือไม่
- ค่าตัวอย่างดูสมเหตุสมผลหรือไม่

คำสั่งที่ใช้บ่อย ได้แก่

- `.head()`
- `.tail()`
- `.sample()`

### ดูข้อมูลส่วนบนด้วย `.head()`

โดยค่าเริ่มต้น `.head()` แสดง 5 แถวแรก

In [6]:
mso_df.head()

,วันที่แก้ไขข้อมูลล่าสุด,รหัสครัวเรือน,รหัสประจำบ้าน,วันที่สร้างครัวเรือน,อายุ,เพศ,จังหวัด,ระดับการศึกษา,อาชีพหลัก,ประเภทกลุ่มเป้าหมาย,รหัส cm,หน่วยงานของ cm
0,2023-06-19 13:40:56.585,648ff7d6b64d06b6b0dc7f02,<NA>,2023-06-19 13:38:14.881,3,หญิง,อุดรธานี,ไม่ได้เรียนหนังสือ,เกษตรกรรม (พืช ปศุสัตว์ ประมง),เด็กเล็ก,cm410011,สำนักงานพัฒนาสังคมและความมั่นคงของมนุษย์จังหวั...
1,2022-07-17 18:56:10.147,62d3f7a6d6f101550054198b,70040223124,2022-07-17 18:51:02.889,55,หญิง,ราชบุรี,ไม่ได้เรียนหนังสือ,NaN,วัยผู้ใหญ่/วัยแรงงาน,cm700018,ศูนย์คุ้มครองคนไร้ที่พึ่งราชบุรี
2,2024-01-25 13:34:46.941,62a05c9df5674ecdd3805787,<NA>,2022-08-06 15:23:57.850,33,หญิง,หนองคาย,มัธยมศึกษาตอนต้น,NaN,วัยผู้ใหญ่/วัยแรงงาน,cm430010,สำนักงานพัฒนาสังคมและความมั่นคงของมนุษย์จังหวั...
3,2024-03-22 14:12:39.290,62908e378fa67bf5ad7d5555,<NA>,2022-05-27 15:39:19.197,5,หญิง,อุบลราชธานี,ไม่ได้เรียนหนังสือ,NaN,เด็กเล็ก,cm340009,สำนักงานพัฒนาสังคมและความมั่นคงของมนุษย์จังหวั...
4,2023-03-15 09:27:11.763,6319b26ca3a37508384e36fc,<NA>,2022-08-09 16:14:20.652,3,ชาย,นครราชสีมา,ไม่ได้เรียนหนังสือ,NaN,เด็กเล็ก,cm300020,ศูนย์บริการคนพิการ จังหวัดนครราชสีมา


In [7]:
mso_df.head(10)

,วันที่แก้ไขข้อมูลล่าสุด,รหัสครัวเรือน,รหัสประจำบ้าน,วันที่สร้างครัวเรือน,อายุ,เพศ,จังหวัด,ระดับการศึกษา,อาชีพหลัก,ประเภทกลุ่มเป้าหมาย,รหัส cm,หน่วยงานของ cm
0,2023-06-19 13:40:56.585,648ff7d6b64d06b6b0dc7f02,<NA>,2023-06-19 13:38:14.881,3,หญิง,อุดรธานี,ไม่ได้เรียนหนังสือ,เกษตรกรรม (พืช ปศุสัตว์ ประมง),เด็กเล็ก,cm410011,สำนักงานพัฒนาสังคมและความมั่นคงของมนุษย์จังหวั...
1,2022-07-17 18:56:10.147,62d3f7a6d6f101550054198b,70040223124,2022-07-17 18:51:02.889,55,หญิง,ราชบุรี,ไม่ได้เรียนหนังสือ,NaN,วัยผู้ใหญ่/วัยแรงงาน,cm700018,ศูนย์คุ้มครองคนไร้ที่พึ่งราชบุรี
2,2024-01-25 13:34:46.941,62a05c9df5674ecdd3805787,<NA>,2022-08-06 15:23:57.850,33,หญิง,หนองคาย,มัธยมศึกษาตอนต้น,NaN,วัยผู้ใหญ่/วัยแรงงาน,cm430010,สำนักงานพัฒนาสังคมและความมั่นคงของมนุษย์จังหวั...
3,2024-03-22 14:12:39.290,62908e378fa67bf5ad7d5555,<NA>,2022-05-27 15:39:19.197,5,หญิง,อุบลราชธานี,ไม่ได้เรียนหนังสือ,NaN,เด็กเล็ก,cm340009,สำนักงานพัฒนาสังคมและความมั่นคงของมนุษย์จังหวั...
4,2023-03-15 09:27:11.763,6319b26ca3a37508384e36fc,<NA>,2022-08-09 16:14:20.652,3,ชาย,นครราชสีมา,ไม่ได้เรียนหนังสือ,NaN,เด็กเล็ก,cm300020,ศูนย์บริการคนพิการ จังหวัดนครราชสีมา
5,2023-03-16 13:23:07.617,63ad11f0049951d53b429c13,<NA>,2022-12-29 11:05:04.736,39,ชาย,ประจวบคีรีขันธ์,มัธยมศึกษาตอนต้น,ธุรกิจส่วนตัว/ค้าขาย/เจ้าของกิจการ,วัยผู้ใหญ่/วัยแรงงาน,cm770005,สำนักงานพัฒนาสังคมและความมั่นคงของมนุษย์จังหวั...
6,2022-12-22 10:35:26.057,63a3d037cdf0fb5ddc8ba1f9,<NA>,2022-12-22 10:34:15.185,4,หญิง,อุดรธานี,ไม่ได้เรียนหนังสือ,NaN,เด็กเล็ก,cm410006,สำนักงานพัฒนาสังคมและความมั่นคงของมนุษย์จังหวั...
7,2023-01-02 11:34:02.089,63d9eb642617b2b7aa05ea6c,<NA>,2023-01-02 11:32:36.354,68,NaN,นครศรีธรรมราช,NaN,ธุรกิจส่วนตัว/ค้าขาย/เจ้าของกิจการ,ผู้สูงอายุ,cm800045,ศูนย์คุ้มครองคนไร้ที่พึ่ง จ.นครศรีธรรมราช
8,2022-09-22 14:23:42.121,632c086a720e4eb0fded2524,25120211445,2022-09-22 14:02:02.435,90,หญิง,สระแก้ว,ประถมศึกษา,NaN,ผู้สูงอายุ,cm270003,สำนักงานพัฒนาสังคมและความมั่นคงของมนุษย์จังหวั...
9,2022-12-23 10:02:55.712,63a519a7cdf0fb5ddc8bb44a,<NA>,2022-12-23 09:59:51.518,6,หญิง,บุรีรัมย์,ไม่ได้เรียนหนังสือ,NaN,เด็ก,cm310013,บ้านพักเด็กและครอบครัวจังหวัดบุรีรัมย์


### ดูข้อมูลส่วนท้ายด้วย `.tail()`

เหมาะสำหรับตรวจว่าข้อมูลท้ายไฟล์ถูกอ่านเข้ามาปกติหรือไม่

In [8]:
mso_df.tail()

,วันที่แก้ไขข้อมูลล่าสุด,รหัสครัวเรือน,รหัสประจำบ้าน,วันที่สร้างครัวเรือน,อายุ,เพศ,จังหวัด,ระดับการศึกษา,อาชีพหลัก,ประเภทกลุ่มเป้าหมาย,รหัส cm,หน่วยงานของ cm
95,2025-07-29 14:33:32.178,688878eaa933a405b21cc136,<NA>,2025-07-29 14:31:54.610,59,NaN,ชัยภูมิ,NaN,เกษตรกรรม (พืช ปศุสัตว์ ประมง),วัยผู้ใหญ่/วัยแรงงาน,cm360060,ศูนย์คุ้มครองคนไร้ที่พึ่ง จ.ชัยภูมิ
96,2023-03-17 19:08:15.489,628b008c8fa67bf5ad7d221b,<NA>,2022-05-23 10:33:32.270,79,NaN,มหาสารคาม,NaN,NaN,ผู้สูงอายุ,cm440005,สำนักงานพัฒนาสังคมและความมั่นคงของมนุษย์จังหวั...
97,2023-05-01 09:19:26.395,63b6325b2617b2b7aa04dc0b,25120251099,2023-05-01 09:13:47.520,51,หญิง,สระแก้ว,ประถมศึกษา,รับจ้างทั่วไป,วัยผู้ใหญ่/วัยแรงงาน,cm270001,สำนักงานพัฒนาสังคมและความมั่นคงของมนุษย์จังหวั...
98,2025-01-04 14:05:25.955,63e1d0d92617b2b7aa06096d,40160104483,2023-07-02 11:17:29.369,81,ชาย,ขอนแก่น,ไม่ได้เรียนหนังสือ,NaN,ผู้สูงอายุ,cm400016,ศูนย์เรียนรู้การพัฒนาสตรีและครอบครัวรัตนาภา จั...
99,2025-03-18 15:17:33.047,67d92537a933a405b21c2e77,<NA>,2025-03-18 14:48:07.735,70,หญิง,นครราชสีมา,ประถมศึกษา,เกษตรกรรม (พืช ปศุสัตว์ ประมง),ผู้สูงอายุ,cm300037,บ้านพักเด็กและครอบครัวจังหวัดนครราชสีมา


### ดูข้อมูลแบบสุ่มด้วย `.sample()`

การดูเพียงส่วนต้นและท้ายอาจไม่ครอบคลุมรูปแบบข้อมูลทั้งหมด

`.sample()` ช่วยสุ่มแถวจากตำแหน่งต่าง ๆ ใน DataFrame

In [9]:
mso_df.sample(
    5,
    random_state=42,
)

,วันที่แก้ไขข้อมูลล่าสุด,รหัสครัวเรือน,รหัสประจำบ้าน,วันที่สร้างครัวเรือน,อายุ,เพศ,จังหวัด,ระดับการศึกษา,อาชีพหลัก,ประเภทกลุ่มเป้าหมาย,รหัส cm,หน่วยงานของ cm
83,2023-09-05 16:01:43.789,6284a65d8fa67bf5ad7d0949,<NA>,2022-05-18 14:55:09.017,37,NaN,เพชรบุรี,NaN,NaN,วัยผู้ใหญ่/วัยแรงงาน,cm760001,สำนักงานพัฒนาสังคมและความมั่นคงของมนุษย์จังหวั...
53,2024-02-04 11:36:52.215,6476aad3611fdbad2925b62f,<NA>,2023-05-31 09:02:59.706,35,หญิง,สมุทรสงคราม,ประถมศึกษา,NaN,วัยผู้ใหญ่/วัยแรงงาน,cm750009,สำนักงานพัฒนาสังคมและความมั่นคงของมนุษย์จังหวั...
70,2023-09-02 14:54:48.248,6386a834ed6d46a6baea7954,<NA>,2022-11-30 07:47:48.806,30,หญิง,อุดรธานี,มัธยมศึกษาตอนต้น,รับจ้างทั่วไป,วัยผู้ใหญ่/วัยแรงงาน,cm410008,สำนักงานพัฒนาสังคมและความมั่นคงของมนุษย์จังหวั...
45,2023-08-23 13:23:18.052,64e5a537b64d06b6b0ddeee6,<NA>,2023-08-23 13:20:39.596,64,หญิง,ชุมพร,ประถมศึกษา,รับจ้างทั่วไป,ผู้สูงอายุ,cm860009,ศูนย์บริการคนพิการ จังหวัด ชุมพร
44,2023-08-07 13:03:43.301,64a8fbc9b64d06b6b0dcf3f1,<NA>,2023-08-07 13:01:45.221,35,หญิง,ศรีสะเกษ,ประถมศึกษา,เกษตรกรรม (พืช ปศุสัตว์ ประมง),วัยผู้ใหญ่/วัยแรงงาน,cm330005,สำนักงานพัฒนาสังคมและความมั่นคงของมนุษย์จังหวั...


`random_state=42` ทำให้ได้ตัวอย่างเดิมทุกครั้งที่รัน

เหมาะกับการสอน การตรวจสอบร่วมกัน และงานที่ต้องการผลลัพธ์ซ้ำได้

## 4. ตรวจสอบขนาด ชื่อคอลัมน์ และ Index

ก่อนวิเคราะห์ควรทราบว่า

- ข้อมูลมีกี่แถว
- มีกี่คอลัมน์
- มีคอลัมน์ใดบ้าง
- index มีรูปแบบอย่างไร

### ตรวจสอบขนาดด้วย `.shape`

ผลลัพธ์อยู่ในรูปแบบ

```python
(number_of_rows, number_of_columns)
```

In [10]:
mso_df.shape

(100, 12)

In [11]:
number_of_rows = mso_df.shape[0]
number_of_columns = mso_df.shape[1]

print("จำนวนแถว:", number_of_rows)
print("จำนวนคอลัมน์:", number_of_columns)

จำนวนแถว: 100
จำนวนคอลัมน์: 12


จำนวนแถวและคอลัมน์ควรนำไปเปรียบเทียบกับ

- จำนวนที่เจ้าของข้อมูลแจ้ง
- จำนวนในรายงานต้นทาง
- จำนวนจากรอบก่อนหน้า
- จำนวนที่คาดจากเงื่อนไขการเลือกข้อมูล

### ตรวจสอบชื่อคอลัมน์ด้วย `.columns`

In [12]:
mso_df.columns

Index(['วันที่แก้ไขข้อมูลล่าสุด', 'รหัสครัวเรือน', 'รหัสประจำบ้าน',
       'วันที่สร้างครัวเรือน', 'อายุ', 'เพศ', 'จังหวัด', 'ระดับการศึกษา',
       'อาชีพหลัก', 'ประเภทกลุ่มเป้าหมาย', 'รหัส cm', 'หน่วยงานของ cm'],
      dtype='str')

In [13]:
mso_df.columns.tolist()

['วันที่แก้ไขข้อมูลล่าสุด',
 'รหัสครัวเรือน',
 'รหัสประจำบ้าน',
 'วันที่สร้างครัวเรือน',
 'อายุ',
 'เพศ',
 'จังหวัด',
 'ระดับการศึกษา',
 'อาชีพหลัก',
 'ประเภทกลุ่มเป้าหมาย',
 'รหัส cm',
 'หน่วยงานของ cm']

การตรวจชื่อคอลัมน์ช่วยให้ทราบว่า

- คอลัมน์ถูกอ่านเข้ามาครบหรือไม่
- ชื่อตรงกับ Data Dictionary หรือไม่
- มีช่องว่างหรืออักขระที่ไม่ต้องการหรือไม่
- ชื่อตรงกับชื่อที่จะใช้ในโค้ดหรือไม่

### ตรวจสอบ Index

In [14]:
mso_df.index

RangeIndex(start=0, stop=100, step=1)

หากไม่ได้กำหนด index เอง pandas จะสร้าง `RangeIndex` เริ่มจาก `0`

Index ไม่จำเป็นต้องเป็นรหัสประจำตัวของข้อมูลเสมอไป แต่เป็น label ที่ pandas ใช้อ้างอิงแถว

## 5. ตรวจสอบชนิดข้อมูลและภาพรวม DataFrame

ชนิดข้อมูลมีผลต่อการคำนวณ การกรอง และการตรวจสอบข้อมูล

คำสั่งหลัก ได้แก่

- `.dtypes`
- `.info()`

In [15]:
mso_df.dtypes

วันที่แก้ไขข้อมูลล่าสุด    datetime64[us]
รหัสครัวเรือน                      string
รหัสประจำบ้าน                      string
วันที่สร้างครัวเรือน       datetime64[us]
อายุ                                int64
เพศ                                   str
จังหวัด                               str
ระดับการศึกษา                         str
อาชีพหลัก                             str
ประเภทกลุ่มเป้าหมาย                   str
รหัส cm                            string
หน่วยงานของ cm                        str
dtype: object

ชนิดข้อมูลที่พบบ่อยใน pandas ได้แก่

| dtype | ความหมาย | ตัวอย่าง |
|---|---|---|
| `object` | มักเป็นข้อความหรือข้อมูลหลายชนิด | จังหวัด อาชีพ |
| `string` | ข้อความแบบ pandas | รหัสครัวเรือน |
| `int64` | จำนวนเต็ม | จำนวนรายการ |
| `float64` | ทศนิยมหรือจำนวนที่มีค่าว่าง | อายุ รายได้ |
| `datetime64[ns]` | วันที่และเวลา | วันที่สร้างข้อมูล |
| `bool` | จริงหรือเท็จ | ผ่านหรือไม่ผ่าน |

ข้อควรระวัง:

- รหัสควรเป็นข้อความ ไม่ใช่ตัวเลข
- วันที่ควรเป็น `datetime` เมื่อต้องกรองหรือคำนวณช่วงเวลา
- คอลัมน์ตัวเลขที่มีค่าว่างอาจถูกอ่านเป็น `float64`

### ดูภาพรวมด้วย `.info()`

In [16]:
mso_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 12 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   วันที่แก้ไขข้อมูลล่าสุด  100 non-null    datetime64[us]
 1   รหัสครัวเรือน            100 non-null    string        
 2   รหัสประจำบ้าน            24 non-null     string        
 3   วันที่สร้างครัวเรือน     100 non-null    datetime64[us]
 4   อายุ                     100 non-null    int64         
 5   เพศ                      77 non-null     str           
 6   จังหวัด                  100 non-null    str           
 7   ระดับการศึกษา            77 non-null     str           
 8   อาชีพหลัก                39 non-null     str           
 9   ประเภทกลุ่มเป้าหมาย      100 non-null    str           
 10  รหัส cm                  100 non-null    string        
 11  หน่วยงานของ cm           100 non-null    str           
dtypes: datetime64[us](2), int64(1), str(6), string(3

`.info()` แสดงข้อมูล เช่น

- จำนวนแถว
- จำนวนคอลัมน์
- ชื่อคอลัมน์
- จำนวนค่าที่ไม่ว่าง
- ชนิดข้อมูล
- ขนาดหน่วยความจำโดยประมาณ

คำสั่งนี้ช่วยตรวจพบคอลัมน์ที่มีค่าว่างหรือชนิดข้อมูลผิดคาดได้อย่างรวดเร็ว

## 6. ทำความเข้าใจค่าสถิติพื้นฐาน

ค่าสถิติพื้นฐานช่วยให้เห็นการกระจายและช่วงค่าของข้อมูลตัวเลข

ค่าที่พบใน `describe()` ได้แก่

| ค่าสถิติ | ความหมาย | ประโยชน์ในการตรวจข้อมูล |
|---|---|---|
| `count` | จำนวนค่าที่ไม่ว่าง | เปรียบเทียบกับจำนวนแถว |
| `mean` | ค่าเฉลี่ย | ดูค่ากลางโดยรวม |
| `std` | ส่วนเบี่ยงเบนมาตรฐาน | ดูการกระจาย |
| `min` | ค่าน้อยที่สุด | ตรวจค่าต่ำผิดปกติ |
| `25%` | ควอร์ไทล์ที่ 1 | ดูช่วงล่างของข้อมูล |
| `50%` | ค่ามัธยฐาน | ดูค่ากลางที่ทนต่อค่าผิดปกติ |
| `75%` | ควอร์ไทล์ที่ 3 | ดูช่วงบนของข้อมูล |
| `max` | ค่าสูงที่สุด | ตรวจค่าสูงผิดปกติ |

### ค่าเฉลี่ยและค่ามัธยฐาน

ค่าเฉลี่ยคำนวณจากผลรวมหารด้วยจำนวนข้อมูล จึงได้รับผลกระทบจากค่าที่สูงหรือต่ำผิดปกติ

ค่ามัธยฐานคือค่าตรงกลางเมื่อเรียงข้อมูล และมักทนต่อค่าผิดปกติได้ดีกว่า

In [17]:
example_ages = pd.Series(
    [20, 21, 22, 23, 80]
)

print(
    "mean:",
    example_ages.mean(),
)

print(
    "median:",
    example_ages.median(),
)

mean: 33.2
median: 22.0


ถ้า `mean` และ `median` ต่างกันมาก อาจเป็นสัญญาณว่าข้อมูลมีค่าที่สูงหรือต่ำผิดปกติ แต่ยังไม่เพียงพอที่จะสรุปว่าค่านั้นผิด

### ส่วนเบี่ยงเบนมาตรฐาน

`std` ย่อมาจาก Standard Deviation

อธิบายอย่างง่ายว่าเป็นค่าที่ช่วยบอกว่าข้อมูลกระจายออกจากค่าเฉลี่ยมากน้อยเพียงใด

- `std` ต่ำ: ค่าส่วนใหญ่อยู่ใกล้กัน
- `std` สูง: ค่ากระจายกว้าง หรืออาจมีค่าผิดปกติ

In [18]:
ages_close = pd.Series(
    [30, 31, 32, 33, 34]
)

ages_spread = pd.Series(
    [10, 20, 30, 40, 100]
)

print(
    "std ของข้อมูลที่ใกล้กัน:",
    ages_close.std(),
)

print(
    "std ของข้อมูลที่กระจายมาก:",
    ages_spread.std(),
)

std ของข้อมูลที่ใกล้กัน: 1.5811388300841898
std ของข้อมูลที่กระจายมาก: 35.35533905932738


### Percentile

Percentile แสดงตำแหน่งของข้อมูลเมื่อเรียงจากน้อยไปมาก

- `25%` หมายถึง 25% ของข้อมูลมีค่าน้อยกว่าหรือเท่ากับค่านี้
- `50%` คือค่ามัธยฐาน
- `75%` หมายถึง 75% ของข้อมูลมีค่าน้อยกว่าหรือเท่ากับค่านี้

Percentile ช่วยให้เห็นว่าข้อมูลส่วนใหญ่กระจุกตัวอยู่ในช่วงใด

## 7. สรุปข้อมูลตัวเลขด้วย `describe()`

โดยค่าเริ่มต้น `describe()` สรุปเฉพาะคอลัมน์ตัวเลข

In [19]:
mso_df.describe()

,วันที่แก้ไขข้อมูลล่าสุด,วันที่สร้างครัวเรือน,อายุ
count,100,100,100.000000
mean,2023-09-24 19:42:21.373390,2023-03-23 05:44:38.681930,42.200000
min,2022-02-08 08:55:15.688000,2022-01-06 15:21:51.132000,1.000000
25%,2023-02-27 04:55:33.963750,2022-08-17 10:34:13.151750,26.000000
50%,2023-08-13 14:51:07.477000,2022-12-23 22:54:09.762500,44.500000
75%,2024-03-16 22:49:22.254250,2023-08-30 12:40:05.888000,59.000000
max,2025-11-03 14:02:41.077000,2025-07-29 14:31:54.610000,103.000000
std,NaN,NaN,25.063153


ในข้อมูลตัวอย่าง คอลัมน์ตัวเลขสำคัญคือ `อายุ`

In [20]:
mso_df["อายุ"].describe()

count    100.000000
mean      42.200000
std       25.063153
min        1.000000
25%       26.000000
50%       44.500000
75%       59.000000
max      103.000000
Name: อายุ, dtype: float64

ควรสังเกตเป็นพิเศษว่า

- `count` เท่ากับจำนวนแถวหรือไม่
- `min` ต่ำผิดปกติหรือไม่
- `max` สูงผิดปกติหรือไม่
- `mean` และ `50%` แตกต่างกันมากหรือไม่
- ช่วงระหว่าง `25%` และ `75%` สมเหตุสมผลหรือไม่

หากพบค่าที่น่าสงสัย ควรบันทึกเป็นประเด็นเพื่อตรวจสอบ ไม่ควรแก้ไขทันทีโดยไม่มีหลักฐาน

### สรุปทุกชนิดข้อมูลด้วย `include="all"`

หากต้องการให้ `describe()` รวมคอลัมน์ข้อความและวันที่ สามารถใช้

In [21]:
mso_df.describe(
    include="all"
)

,วันที่แก้ไขข้อมูลล่าสุด,รหัสครัวเรือน,รหัสประจำบ้าน,วันที่สร้างครัวเรือน,อายุ,เพศ,จังหวัด,ระดับการศึกษา,อาชีพหลัก,ประเภทกลุ่มเป้าหมาย,รหัส cm,หน่วยงานของ cm
count,100,100,24,100,100.000000,77,100,77,39,100,100,100
unique,NaN,100,24,NaN,NaN,2,48,4,4,5,94,74
top,NaN,648ff7d6b64d06b6b0dc7f02,70040223124,NaN,NaN,หญิง,นครศรีธรรมราช,ประถมศึกษา,รับจ้างทั่วไป,วัยผู้ใหญ่/วัยแรงงาน,cm610009,สำนักงานพัฒนาสังคมและความมั่นคงของมนุษย์จังหวั...
freq,NaN,1,1,NaN,NaN,45,8,42,26,54,2,7
mean,2023-09-24 19:42:21.373390,NaN,NaN,2023-03-23 05:44:38.681930,42.200000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
min,2022-02-08 08:55:15.688000,NaN,NaN,2022-01-06 15:21:51.132000,1.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25%,2023-02-27 04:55:33.963750,NaN,NaN,2022-08-17 10:34:13.151750,26.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
50%,2023-08-13 14:51:07.477000,NaN,NaN,2022-12-23 22:54:09.762500,44.500000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
75%,2024-03-16 22:49:22.254250,NaN,NaN,2023-08-30 12:40:05.888000,59.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
max,2025-11-03 14:02:41.077000,NaN,NaN,2025-07-29 14:31:54.610000,103.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN


สำหรับคอลัมน์เชิงหมวดหมู่ pandas อาจแสดง

- `count` — จำนวนค่าที่ไม่ว่าง
- `unique` — จำนวนค่าที่ไม่ซ้ำ
- `top` — ค่าที่พบมากที่สุด
- `freq` — จำนวนครั้งที่ค่าที่พบบ่อยที่สุดปรากฏ

อย่างไรก็ตาม ควรใช้ `value_counts()` เพิ่มเติมเพื่อดูรายละเอียดของแต่ละหมวดหมู่

## 8. ตรวจสอบค่าที่อยู่นอกช่วง

ข้อมูลตัวเลขควรได้รับการตรวจสอบตามช่วงที่สมเหตุสมผลหรือกฎของงาน

ตัวอย่างนี้กำหนดช่วงอายุที่ยอมรับได้เป็น `0–100` ปี

ค่าที่ต้องตรวจสอบคือ

```python
อายุ < 0 หรือ อายุ > 100
```

In [23]:
unusual_age_condition = (
    (mso_df["อายุ"] < 0)
    | (mso_df["อายุ"] > 100)
)

unusual_age_df = mso_df.loc[
    unusual_age_condition
]

unusual_age_df

,วันที่แก้ไขข้อมูลล่าสุด,รหัสครัวเรือน,รหัสประจำบ้าน,วันที่สร้างครัวเรือน,อายุ,เพศ,จังหวัด,ระดับการศึกษา,อาชีพหลัก,ประเภทกลุ่มเป้าหมาย,รหัส cm,หน่วยงานของ cm
64,2023-06-13 19:06:43.348,63230d7ddd08ee35263d3fc5,<NA>,2022-09-15 18:33:17.727,103,NaN,นครราชสีมา,NaN,NaN,ผู้สูงอายุ,cm300009,สำนักงานพัฒนาสังคมและความมั่นคงของมนุษย์จังหวั...


In [24]:
unusual_age_count = (
    unusual_age_df.shape[0]
)

print(
    "จำนวนรายการที่อายุผิดปกติ:",
    unusual_age_count,
)


จำนวนรายการที่อายุผิดปกติ: 1


การอยู่นอกช่วงไม่ได้หมายความว่าต้องลบทันที

ควรตรวจสอบเพิ่มเติมว่า

- เป็นข้อผิดพลาดจากการบันทึกหรือไม่
- หน่วยของข้อมูลถูกต้องหรือไม่
- ค่าเกิดจากกฎหรือกรณีพิเศษหรือไม่
- สามารถตรวจสอบกับแหล่งต้นทางได้หรือไม่

## 9. ตรวจสอบค่าว่าง

`isna()` ตรวจสอบว่าแต่ละตำแหน่งเป็นค่าว่างหรือไม่

ผลลัพธ์เป็น Boolean DataFrame

In [25]:
mso_df.isna()

,วันที่แก้ไขข้อมูลล่าสุด,รหัสครัวเรือน,รหัสประจำบ้าน,วันที่สร้างครัวเรือน,อายุ,เพศ,จังหวัด,ระดับการศึกษา,อาชีพหลัก,ประเภทกลุ่มเป้าหมาย,รหัส cm,หน่วยงานของ cm
0,False,False,True,False,False,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False,True,False,False,False
2,False,False,True,False,False,False,False,False,True,False,False,False
3,False,False,True,False,False,False,False,False,True,False,False,False
4,False,False,True,False,False,False,False,False,True,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...
95,False,False,True,False,False,True,False,True,False,False,False,False
96,False,False,True,False,False,True,False,True,True,False,False,False
97,False,False,False,False,False,False,False,False,False,False,False,False
98,False,False,False,False,False,False,False,False,True,False,False,False


การดูทั้งตารางอาจอ่านยาก จึงมักใช้ร่วมกับ `.sum()` เพื่อหาจำนวนค่าว่างของแต่ละคอลัมน์

In [26]:
mso_df.isna().sum()

วันที่แก้ไขข้อมูลล่าสุด     0
รหัสครัวเรือน               0
รหัสประจำบ้าน              76
วันที่สร้างครัวเรือน        0
อายุ                        0
เพศ                        23
จังหวัด                     0
ระดับการศึกษา              23
อาชีพหลัก                  61
ประเภทกลุ่มเป้าหมาย         0
รหัส cm                     0
หน่วยงานของ cm              0
dtype: int64

### จำนวนและเปอร์เซ็นต์ค่าว่าง

จำนวนค่าว่างเพียงอย่างเดียวอาจไม่สะท้อนความรุนแรงของปัญหา

ตัวอย่างเช่น

- ว่าง 100 แถวจาก 1,000 แถว เท่ากับ 10%
- ว่าง 100 แถวจาก 1,000,000 แถว เท่ากับ 0.01%

จึงควรดูทั้งจำนวนและเปอร์เซ็นต์

In [27]:
missing_count = (
    mso_df.isna().sum()
)

missing_percent = (
    mso_df.isna().mean()
    * 100
)

missing_percent

วันที่แก้ไขข้อมูลล่าสุด     0.0
รหัสครัวเรือน               0.0
รหัสประจำบ้าน              76.0
วันที่สร้างครัวเรือน        0.0
อายุ                        0.0
เพศ                        23.0
จังหวัด                     0.0
ระดับการศึกษา              23.0
อาชีพหลัก                  61.0
ประเภทกลุ่มเป้าหมาย         0.0
รหัส cm                     0.0
หน่วยงานของ cm              0.0
dtype: float64

### สร้างตารางสรุปค่าว่าง

In [28]:
missing_summary = pd.DataFrame(
    {
        "column": mso_df.columns,
        "missing_count": (
            mso_df
            .isna()
            .sum()
            .values
        ),
        "missing_percent": (
            mso_df
            .isna()
            .mean()
            .mul(100)
            .round(2)
            .values
        ),
    }
)

missing_summary

,column,missing_count,missing_percent
0,วันที่แก้ไขข้อมูลล่าสุด,0,0.0
1,รหัสครัวเรือน,0,0.0
2,รหัสประจำบ้าน,76,76.0
3,วันที่สร้างครัวเรือน,0,0.0
4,อายุ,0,0.0
5,เพศ,23,23.0
6,จังหวัด,0,0.0
7,ระดับการศึกษา,23,23.0
8,อาชีพหลัก,61,61.0
9,ประเภทกลุ่มเป้าหมาย,0,0.0


In [29]:
missing_summary.sort_values(
    "missing_percent",
    ascending=False,
)

,column,missing_count,missing_percent
2,รหัสประจำบ้าน,76,76.0
8,อาชีพหลัก,61,61.0
5,เพศ,23,23.0
7,ระดับการศึกษา,23,23.0
3,วันที่สร้างครัวเรือน,0,0.0
1,รหัสครัวเรือน,0,0.0
0,วันที่แก้ไขข้อมูลล่าสุด,0,0.0
4,อายุ,0,0.0
6,จังหวัด,0,0.0
9,ประเภทกลุ่มเป้าหมาย,0,0.0


การพบค่าว่างนำไปสู่คำถาม เช่น

- ค่าว่างเกิดจากอะไร
- เป็นค่าว่างที่ยอมรับได้หรือไม่
- เป็นข้อมูลที่ไม่ได้เก็บหรือไม่เกี่ยวข้อง
- ควรเติมค่า ลบแถว ลบคอลัมน์ หรือคงไว้
- คอลัมน์สำคัญมีค่าว่างมากเกินไปหรือไม่

การตัดสินใจแก้ไขจะดำเนินการในขั้นตอน Data Cleaning

## 10. ตรวจสอบข้อมูลซ้ำ

ข้อมูลซ้ำมีได้หลายระดับ เช่น

1. ทั้งแถวซ้ำกันทุกคอลัมน์
2. รหัสสำคัญซ้ำ
3. รายการเดียวกันถูกอัปเดตหลายครั้ง
4. หนึ่งหน่วยข้อมูลมีหลายรายการตามธรรมชาติ

จึงต้องตรวจทั้งจำนวนและความหมายเชิงธุรกิจ

### ตรวจแถวที่ซ้ำทุกคอลัมน์

In [30]:
mso_df.duplicated()

0     False
1     False
2     False
3     False
4     False
      ...  
95    False
96    False
97    False
98    False
99    False
Length: 100, dtype: bool

In [31]:
duplicate_row_count = (
    mso_df.duplicated().sum()
)

duplicate_row_count

np.int64(0)

### ตรวจข้อมูลซ้ำจากรหัสสำคัญ

ก่อนตรวจรหัสซ้ำ ควรตรวจค่าว่างในรหัสด้วย เพราะรหัสว่างไม่ควรถูกตีความเหมือนรหัสที่มีค่าจริง

In [32]:
household_id_missing_count = (
    mso_df["รหัสครัวเรือน"]
    .isna()
    .sum()
)

household_id_missing_count

np.int64(0)

In [33]:
duplicated_household_count = (
    mso_df["รหัสครัวเรือน"]
    .duplicated()
    .sum()
)

duplicated_household_count

np.int64(0)

In [34]:
duplicated_household_condition = (
    mso_df["รหัสครัวเรือน"]
    .duplicated(keep=False)
)

duplicated_household_df = (
    mso_df.loc[
        duplicated_household_condition
    ]
)

duplicated_household_df

,วันที่แก้ไขข้อมูลล่าสุด,รหัสครัวเรือน,รหัสประจำบ้าน,วันที่สร้างครัวเรือน,อายุ,เพศ,จังหวัด,ระดับการศึกษา,อาชีพหลัก,ประเภทกลุ่มเป้าหมาย,รหัส cm,หน่วยงานของ cm


ในตัวอย่างนี้ไม่พบข้อมูลซ้ำ ซึ่งถือว่าเป็นเรื่องที่ดี แต่ในบางกรณีอาจจะมีการพบรหัสซ้ำได้ 

การพบรหัสซ้ำไม่ได้หมายความว่าข้อมูลผิดเสมอไป ตัวอย่างเหตุผล ได้แก่

- หนึ่งครัวเรือนมีสมาชิกหลายคน
- หนึ่งรหัสมีหลายรายการจากการอัปเดต
- ข้อมูลมาจากหลายระบบ
- เป็นข้อมูลซ้ำที่ควรถูกลบจริง

จึงต้องตรวจสอบนิยามของหนึ่งแถวและความหมายของรหัสก่อนตัดสินใจ


## 11. ตรวจสอบข้อมูลเชิงหมวดหมู่

ข้อมูลเชิงหมวดหมู่คือข้อมูลที่แบ่งเป็นกลุ่มหรือประเภท เช่น

- จังหวัด
- เพศ
- ระดับการศึกษา
- อาชีพหลัก
- ประเภทกลุ่มเป้าหมาย

การตรวจสอบควรเน้น

- จำนวนของแต่ละหมวดหมู่
- จำนวนค่าที่ไม่ซ้ำ
- ค่าที่สะกดไม่สอดคล้องกัน
- ค่าว่างหรือค่าที่ใช้แทนค่าว่าง

### นับจำนวนด้วย `value_counts()`

In [35]:
mso_df[
    "จังหวัด"
].value_counts()

จังหวัด
นครศรีธรรมราช      8
อุดรธานี           7
เชียงใหม่          5
หนองคาย            4
นครราชสีมา         4
บุรีรัมย์          4
ขอนแก่น            4
นครนายก            4
อุทัยธานี          4
สระแก้ว            3
นครพนม             3
สุราษฎร์ธานี       3
ชุมพร              3
ราชบุรี            2
ปราจีนบุรี         2
เชียงราย           2
ตาก                2
สงขลา              2
เพชรบุรี           2
มหาสารคาม          2
กาญจนบุรี          2
ชัยภูมิ            2
อุบลราชธานี        1
ประจวบคีรีขันธ์    1
ยโสธร              1
นครปฐม             1
สุพรรณบุรี         1
ระนอง              1
เพชรบูรณ์          1
นราธิวาส           1
สิงห์บุรี          1
พิษณุโลก           1
กาฬสินธุ์          1
ศรีสะเกษ           1
ลพบุรี             1
ระยอง              1
จันทบุรี           1
พังงา              1
สมุทรสงคราม        1
บึงกาฬ             1
ชลบุรี             1
มุกดาหาร           1
กระบี่             1
เลย                1
สระบุรี            1
พระนครศรีอยุธยา    1
ยะลา               1
แพร่ 

In [36]:
mso_df[
    "เพศ"
].value_counts()

เพศ
หญิง    45
ชาย     32
Name: count, dtype: int64

In [37]:
mso_df[
    "ระดับการศึกษา"
].value_counts()

ระดับการศึกษา
ประถมศึกษา                42
ไม่ได้เรียนหนังสือ        24
มัธยมศึกษาตอนต้น           7
มัธยมศึกษาตอนปลาย/ปวช.     4
Name: count, dtype: int64

โดยค่าเริ่มต้น `value_counts()` ไม่นับค่าว่าง

หากต้องการรวมค่าว่าง ให้ใช้ `dropna=False`

In [38]:
mso_df[
    "อาชีพหลัก"
].value_counts(
    dropna=False
)

อาชีพหลัก
NaN                                   61
รับจ้างทั่วไป                         26
เกษตรกรรม (พืช ปศุสัตว์ ประมง)        10
ธุรกิจส่วนตัว/ค้าขาย/เจ้าของกิจการ     2
อื่นๆ                                  1
Name: count, dtype: int64

### ตรวจจำนวนค่าที่ไม่ซ้ำด้วย `nunique()`

In [39]:
mso_df[
    "จังหวัด"
].nunique(
    dropna=True
)

48

In [40]:
mso_df[
    "ประเภทกลุ่มเป้าหมาย"
].nunique(
    dropna=True
)

5

### ดูค่าที่ไม่ซ้ำทั้งหมดด้วย `unique()`

เหมาะกับคอลัมน์ที่มีจำนวนหมวดหมู่ไม่มาก

In [41]:
mso_df["เพศ"].unique()

<StringArray>
['หญิง', 'ชาย', nan]
Length: 3, dtype: str

In [42]:
mso_df[
    "ประเภทกลุ่มเป้าหมาย"
].unique()


<StringArray>
['เด็กเล็ก', 'วัยผู้ใหญ่/วัยแรงงาน', 'ผู้สูงอายุ', 'เด็ก', 'เด็กโต/วัยรุ่น']
Length: 5, dtype: str

หากพบค่าที่มีความหมายเดียวกันแต่เขียนต่างกัน เช่น

- `"ชาย"`, `"ช."`, `"Male"`
- `"ไม่ระบุ"`, `"-"`, `"ไม่ทราบ"`
- ช่องว่างและค่าว่าง

ควรบันทึกไว้เพื่อออกแบบกฎการปรับมาตรฐานในขั้นตอนการทำความสะอาดและปรับปรุงข้อมูล

## 12. ตรวจสอบข้อมูลวันที่

คอลัมน์วันที่ควรถูกตรวจสอบว่า

- ถูกอ่านเป็น `datetime` หรือไม่
- มีค่าว่างหรือไม่
- วันที่ต่ำสุดและสูงสุดอยู่ในช่วงที่คาดหรือไม่
- วันที่ระหว่างสองเหตุการณ์มีลำดับสมเหตุสมผลหรือไม่

In [43]:
date_columns = [
    "วันที่สร้างครัวเรือน",
    "วันที่แก้ไขข้อมูลล่าสุด",
]

mso_df[
    date_columns
].dtypes


วันที่สร้างครัวเรือน       datetime64[us]
วันที่แก้ไขข้อมูลล่าสุด    datetime64[us]
dtype: object

In [44]:
mso_df[
    date_columns
].isna().sum()

วันที่สร้างครัวเรือน       0
วันที่แก้ไขข้อมูลล่าสุด    0
dtype: int64

In [45]:
mso_df[
    "วันที่สร้างครัวเรือน"
].min()

Timestamp('2022-01-06 15:21:51.132000')

In [46]:
mso_df[
    "วันที่สร้างครัวเรือน"
].max()

Timestamp('2025-07-29 14:31:54.610000')

In [47]:
mso_df[
    "วันที่แก้ไขข้อมูลล่าสุด"
].min()

Timestamp('2022-02-08 08:55:15.688000')

In [48]:
mso_df[
    "วันที่แก้ไขข้อมูลล่าสุด"
].max()

Timestamp('2025-11-03 14:02:41.077000')

### ตรวจสอบลำดับของวันที่

โดยทั่วไป วันที่แก้ไขล่าสุดไม่ควรอยู่ก่อนวันที่สร้างครัวเรือน

In [49]:
invalid_date_order_condition = (
    mso_df[
        "วันที่แก้ไขข้อมูลล่าสุด"
    ]
    < mso_df[
        "วันที่สร้างครัวเรือน"
    ]
)

invalid_date_order_df = (
    mso_df.loc[
        invalid_date_order_condition
    ]
)

invalid_date_order_df


,วันที่แก้ไขข้อมูลล่าสุด,รหัสครัวเรือน,รหัสประจำบ้าน,วันที่สร้างครัวเรือน,อายุ,เพศ,จังหวัด,ระดับการศึกษา,อาชีพหลัก,ประเภทกลุ่มเป้าหมาย,รหัส cm,หน่วยงานของ cm
15,2023-09-16 13:36:01.293,64fdf7897e0d7cf23231a164,<NA>,2023-11-09 00:06:17.412,2,ชาย,นครปฐม,ไม่ได้เรียนหนังสือ,NaN,เด็กเล็ก,cm730016,บ้านพักเด็กและครอบครัวจังหวัดนครปฐม
19,2022-07-12 10:45:03.770,62fdb4aa064a5083c48848d9,<NA>,2022-08-18 10:40:26.162,75,ชาย,ปราจีนบุรี,ประถมศึกษา,NaN,ผู้สูงอายุ,cm250004,สำนักงานพัฒนาสังคมและความมั่นคงของมนุษย์จังหวั...
20,2022-02-09 15:30:21.934,6306e7b6d8a041e877336c83,61070240168,2022-08-25 10:08:38.004,54,ชาย,อุทัยธานี,ประถมศึกษา,รับจ้างทั่วไป,วัยผู้ใหญ่/วัยแรงงาน,cm610007,ศูนย์คุ้มครองคนไร้ที่พึ่งอุทัยธานี
42,2023-01-13 01:44:01.353,63c036692617b2b7aa051b6f,<NA>,2023-12-01 23:33:45.171,7,หญิง,บุรีรัมย์,ไม่ได้เรียนหนังสือ,NaN,เด็ก,cm310015,ศูนย์บริการคนพิการจังหวัดบุรีรัมย์
52,2023-09-29 15:27:37.255,64fe77607e0d7cf23231a1dd,<NA>,2023-11-09 09:11:44.004,23,NaN,สงขลา,NaN,รับจ้างทั่วไป,วัยผู้ใหญ่/วัยแรงงาน,cm900004,สำนักงานพัฒนาสังคมและความมั่นคงของมนุษย์จังหวั...
73,2023-03-21 03:35:42.442,63bf6df22617b2b7aa0511e7,<NA>,2023-12-01 09:18:26.697,3,หญิง,อุดรธานี,ไม่ได้เรียนหนังสือ,NaN,เด็กเล็ก,cm410019,สำนักงานพัฒนาสังคมและความมั่นคงของมนุษย์จังหวั...
79,2023-03-21 08:59:25.359,63b7aa9c2617b2b7aa04ed78,<NA>,2023-06-01 11:59:08.389,55,หญิง,อุดรธานี,ไม่ได้เรียนหนังสือ,NaN,วัยผู้ใหญ่/วัยแรงงาน,cm410019,สำนักงานพัฒนาสังคมและความมั่นคงของมนุษย์จังหวั...


In [50]:
invalid_date_order_count = (
    invalid_date_order_df.shape[0]
)

print(
    "จำนวนรายการที่ลำดับวันที่ผิดปกติ:",
    invalid_date_order_count,
)

จำนวนรายการที่ลำดับวันที่ผิดปกติ: 7


การตรวจช่วงและลำดับวันที่ช่วยระบุว่า

- ข้อมูลอยู่ในช่วงเวลาที่คาดหรือไม่
- มีวันที่เก่าหรือใหม่ผิดปกติหรือไม่
- วันที่ของเหตุการณ์สัมพันธ์กันอย่างสมเหตุสมผลหรือไม่
- ข้อมูลอาจถูกแปลงปีหรือรูปแบบวันที่ผิดหรือไม่

## 13. สร้างตารางสรุปคุณภาพข้อมูล

ในการทำงานจริง ควรสร้างตารางสรุประดับคอลัมน์ เพื่อช่วยตัดสินใจว่าคอลัมน์ใดควรตรวจสอบหรือทำความสะอาดต่อ

ข้อมูลที่ควรมี เช่น

- ชื่อคอลัมน์
- ชนิดข้อมูล
- จำนวนค่าว่าง
- เปอร์เซ็นต์ค่าว่าง
- จำนวนค่าที่ไม่ซ้ำ

In [51]:
data_quality_summary = pd.DataFrame(
    {
        "column": mso_df.columns,
        "dtype": (
            mso_df
            .dtypes
            .astype(str)
            .values
        ),
        "missing_count": (
            mso_df
            .isna()
            .sum()
            .values
        ),
        "missing_percent": (
            mso_df
            .isna()
            .mean()
            .mul(100)
            .round(2)
            .values
        ),
        "unique_count": (
            mso_df
            .nunique(dropna=True)
            .values
        ),
    }
)

data_quality_summary

,column,dtype,missing_count,missing_percent,unique_count
0,วันที่แก้ไขข้อมูลล่าสุด,datetime64[us],0,0.0,100
1,รหัสครัวเรือน,string,0,0.0,100
2,รหัสประจำบ้าน,string,76,76.0,24
3,วันที่สร้างครัวเรือน,datetime64[us],0,0.0,100
4,อายุ,int64,0,0.0,59
5,เพศ,str,23,23.0,2
6,จังหวัด,str,0,0.0,48
7,ระดับการศึกษา,str,23,23.0,4
8,อาชีพหลัก,str,61,61.0,4
9,ประเภทกลุ่มเป้าหมาย,str,0,0.0,5


In [52]:
data_quality_summary.sort_values(
    "missing_percent",
    ascending=False,
)

,column,dtype,missing_count,missing_percent,unique_count
2,รหัสประจำบ้าน,string,76,76.0,24
8,อาชีพหลัก,str,61,61.0,4
5,เพศ,str,23,23.0,2
7,ระดับการศึกษา,str,23,23.0,4
3,วันที่สร้างครัวเรือน,datetime64[us],0,0.0,100
1,รหัสครัวเรือน,string,0,0.0,100
0,วันที่แก้ไขข้อมูลล่าสุด,datetime64[us],0,0.0,100
4,อายุ,int64,0,0.0,59
6,จังหวัด,str,0,0.0,48
9,ประเภทกลุ่มเป้าหมาย,str,0,0.0,5


ตารางนี้ช่วยให้เห็นภาพรวมว่า

- คอลัมน์ใดมีค่าว่างมาก
- คอลัมน์ใดมีจำนวนค่าที่ไม่ซ้ำสูง
- คอลัมน์ใดอาจเป็นรหัส
- คอลัมน์ใดควรตรวจสอบชนิดข้อมูล
- คอลัมน์ใดอาจไม่เหมาะกับการวิเคราะห์โดยตรง

# แบบฝึกหัดท้ายบท

แบบฝึกหัดท้ายบทนี้จำลองสถานการณ์การตรวจสอบข้อมูลหลังนำเข้าไฟล์

แบบฝึกหัดนี้จะใช้ DataFrame `mso_df` เป็นข้อมูลหลัก

## แบบฝึกหัดที่ 1: สร้าง function สรุปโครงสร้างข้อมูลหลังนำเข้า
หัวหน้าทีมต้องการ function ที่ใช้ดูภาพรวมของ DataFrame หลังจากนำเข้าข้อมูล 

เพื่อให้ทีมสามารถใช้ตรวจไฟล์ใหม่ได้ทุกครั้ง

### ข้อกำหนด
ให้สร้าง function ชื่อ `summarize_dataframe_structure()` โดยมี parameter ดังนี้ 

```python 
def summarize_dataframe_structure(df): 
    ...
```

function นี้ต้องทำงานดังนี้
1. แสดงจำนวนแถวและจำนวนคอลัมน์
2. แสดงรายชื่อ column ทั้งหมด
3. แสดงชนิดข้อมูลของแต่ละ column
4. แสดงตัวอย่างข้อมูล 5 แถวแรก
5. แสดงตัวอย่างข้อมูลแบบสุ่ม 5 แถว โดยใช้ `random_state=42`

หลังจากสร้าง function แล้ว ให้เรียกใช้กับ `mso_df`

In [44]:
# เขียนคำตอบของคุณใน Cell นี้

> ### เฉลยแบบฝึกหัดที่ 1
>
> Function นี้มีหน้าที่แสดงผลเพื่อให้ผู้ใช้ตรวจสอบ DataFrame จึงไม่จำเป็นต้องคืน DataFrame ใหม่
>
> ใช้ `.shape`, `.columns`, `.dtypes`, `.head()` และ `.sample()` ตามข้อกำหนด

In [53]:
def summarize_dataframe_structure(df):
    number_of_rows = df.shape[0]
    number_of_columns = df.shape[1]

    print(
        "จำนวนแถว:",
        number_of_rows,
    )

    print(
        "จำนวนคอลัมน์:",
        number_of_columns,
    )

    print("\nรายชื่อ column:")
    print(df.columns.tolist())

    print("\nชนิดข้อมูล:")
    print(df.dtypes)

    print("\nตัวอย่าง 5 แถวแรก:")
    print(df.head())

    sample_size = min(
        5,
        len(df),
    )

    print("\nตัวอย่างแบบสุ่ม:")
    print(
        df.sample(
            sample_size,
            random_state=42,
        )
    )

In [54]:
summarize_dataframe_structure(
    mso_df
)

จำนวนแถว: 100
จำนวนคอลัมน์: 12

รายชื่อ column:
['วันที่แก้ไขข้อมูลล่าสุด', 'รหัสครัวเรือน', 'รหัสประจำบ้าน', 'วันที่สร้างครัวเรือน', 'อายุ', 'เพศ', 'จังหวัด', 'ระดับการศึกษา', 'อาชีพหลัก', 'ประเภทกลุ่มเป้าหมาย', 'รหัส cm', 'หน่วยงานของ cm']

ชนิดข้อมูล:
วันที่แก้ไขข้อมูลล่าสุด    datetime64[us]
รหัสครัวเรือน                      string
รหัสประจำบ้าน                      string
วันที่สร้างครัวเรือน       datetime64[us]
อายุ                                int64
เพศ                                   str
จังหวัด                               str
ระดับการศึกษา                         str
อาชีพหลัก                             str
ประเภทกลุ่มเป้าหมาย                   str
รหัส cm                            string
หน่วยงานของ cm                        str
dtype: object

ตัวอย่าง 5 แถวแรก:
  วันที่แก้ไขข้อมูลล่าสุด             รหัสครัวเรือน รหัสประจำบ้าน  \
0 2023-06-19 13:40:56.585  648ff7d6b64d06b6b0dc7f02          <NA>   
1 2022-07-17 18:56:10.147  62d3f7a6d6f101550054198b   70040223124   


>การใช้
>
>```python
>sample_size = min(5, len(df))
>```
>
>ช่วยให้ Function ยังทำงานได้เมื่อ DataFrame มีน้อยกว่า 5 แถว

## แบบฝึกหัดที่ 2: สร้าง function สำหรับตรวจสอบค่าว่าง
ทีมต้องการ function ที่สามารถรับ DataFrame ใดก็ได้ 

แล้วสร้างตารางสรุปค่าว่าง เพื่อใช้พิจารณาว่า column ใดควรตรวจสอบต่อ

### ข้อกำหนด
ให้สร้าง function ชื่อ `create_missing_summary()` โดยมี parameter ดังนี้ 

```python 
def create_missing_summary(df): 
    ...
```

function นี้ต้อง return DataFrame ที่มี column ต่อไปนี้
- column
- missing_count
- missing_percent

โดยเรียงลำดับจาก `missing_percent` มากไปน้อย

หลังจากสร้าง function แล้ว ให้เรียกใช้กับ `mso_df` และเก็บผลลัพธ์ไว้ในตัวแปรชื่อ `mso_missing_summary`

In [45]:
# เขียนคำตอบของคุณใน Cell นี้

> ### เฉลยแบบฝึกหัดที่ 2
>
> - `.isna().sum()` ใช้หาจำนวนค่าว่าง
> - `.isna().mean() * 100` ใช้หาเปอร์เซ็นต์ค่าว่าง
> - `.sort_values()` ใช้เรียงคอลัมน์ที่มีปัญหามากที่สุดไว้ด้านบน
> - Function ต้องใช้ `return` เพื่อส่งตารางสรุปกลับมา

In [55]:
def create_missing_summary(df):
    missing_summary = pd.DataFrame(
        {
            "column": df.columns,
            "missing_count": (
                df
                .isna()
                .sum()
                .values
            ),
            "missing_percent": (
                df
                .isna()
                .mean()
                .mul(100)
                .round(2)
                .values
            ),
        }
    )

    missing_summary = (
        missing_summary
        .sort_values(
            "missing_percent",
            ascending=False,
        )
        .reset_index(drop=True)
    )

    return missing_summary

In [56]:
mso_missing_summary = (
    create_missing_summary(
        mso_df
    )
)

mso_missing_summary

,column,missing_count,missing_percent
0,รหัสประจำบ้าน,76,76.0
1,อาชีพหลัก,61,61.0
2,เพศ,23,23.0
3,ระดับการศึกษา,23,23.0
4,วันที่สร้างครัวเรือน,0,0.0
5,รหัสครัวเรือน,0,0.0
6,วันที่แก้ไขข้อมูลล่าสุด,0,0.0
7,อายุ,0,0.0
8,จังหวัด,0,0.0
9,ประเภทกลุ่มเป้าหมาย,0,0.0


## แบบฝึกหัดที่ 3: สร้าง function สำหรับตรวจค่าผิดปกติของ column ตัวเลข
ทีมต้องการตรวจสอบ column ตัวเลข เช่น `อายุ` ว่ามีค่าที่อยู่นอกช่วงสมเหตุสมผลหรือไม่ 

เนื่องจากในอนาคตอาจต้องตรวจ column ตัวเลขอื่น เช่น รายได้ จำนวนสมาชิก หรือคะแนน 

จึงควรสร้าง function ที่รับชื่อ column และช่วงค่าที่ต้องการตรวจได้

### ข้อกำหนด
ให้สร้าง function ชื่อ `check_numeric_range()` โดยมี parameter ดังนี้ 

```python 
def check_numeric_range(df, column, min_value, max_value): 
    ...
```

function นี้ต้องทำงานดังนี้
1. รับ DataFrame จาก df
2. รับชื่อ column ที่ต้องการตรวจจาก column
3. รับค่าต่ำสุดที่ยอมรับได้จาก min_value
4. รับค่าสูงสุดที่ยอมรับได้จาก max_value
5. ถ้า column ไม่มีอยู่ใน DataFrame ให้พิมพ์ข้อความ

>Column not found: <column name>

และ return DataFrame ว่าง

6. ถ้า column มีอยู่จริง ให้เลือก row ที่มีค่าน้อยกว่า min_value หรือมากกว่า max_value
7. return DataFrame ของ row ที่ผิดช่วง

หลังจากสร้าง function แล้ว ให้เรียกใช้เพื่อตรวจ column อายุ โดยกำหนดช่วงที่ยอมรับได้เป็น 0 ถึง 100

`unusual_age_df = check_numeric_range(mso_df, "อายุ", 0, 100)`

In [46]:
# เขียนคำตอบของคุณใน Cell นี้

> ### เฉลยแบบฝึกหัดที่ 3
>
> ตรวจสอบชื่อคอลัมน์ก่อนสร้างเงื่อนไข เพื่อป้องกัน `KeyError`
>
> เงื่อนไขค่าผิดช่วงประกอบด้วย
>
> ```python
> (value < min_value) | (value > max_value)
> ```
>
> ใช้ `.loc` เลือกแถวที่เงื่อนไขเป็นจริง

In [57]:
def check_numeric_range(
    df,
    column,
    min_value,
    max_value,
):
    if column not in df.columns:
        print(
            f"Column not found: {column}"
        )

        return pd.DataFrame()

    unusual_condition = (
        (df[column] < min_value)
        | (df[column] > max_value)
    )

    unusual_df = df.loc[
        unusual_condition
    ].copy()

    return unusual_df

In [59]:
unusual_age_df = (
    check_numeric_range(
        mso_df,
        "อายุ",
        0,
        100,
    )
)

unusual_age_df

,วันที่แก้ไขข้อมูลล่าสุด,รหัสครัวเรือน,รหัสประจำบ้าน,วันที่สร้างครัวเรือน,อายุ,เพศ,จังหวัด,ระดับการศึกษา,อาชีพหลัก,ประเภทกลุ่มเป้าหมาย,รหัส cm,หน่วยงานของ cm
64,2023-06-13 19:06:43.348,63230d7ddd08ee35263d3fc5,<NA>,2022-09-15 18:33:17.727,103,NaN,นครราชสีมา,NaN,NaN,ผู้สูงอายุ,cm300009,สำนักงานพัฒนาสังคมและความมั่นคงของมนุษย์จังหวั...


>Function นี้ไม่จัดให้ค่าว่างเป็นค่าผิดช่วง เพราะการเปรียบเทียบค่าว่างกับตัวเลขให้ผลเป็น `False`
>
>ค่าว่างควรถูกตรวจแยกด้วยตาราง Missing Summary

In [60]:
column_not_found_result = (
    check_numeric_range(
        mso_df,
        "รายได้",
        0,
        1_000_000,
    )
)

column_not_found_result

Column not found: รายได้


""


## แบบฝึกหัดที่ 4: สร้าง function สำหรับตรวจสอบ category หลาย column
ทีมต้องการตรวจสอบค่าที่เป็นไปได้ของ column เชิงหมวดหมู่หลายตัว เช่น จังหวัด เพศ ระดับการศึกษา อาชีพ

และประเภทกลุ่มเป้าหมาย เนื่องจากต้องทำงานนี้ซ้ำกับหลาย dataset จึงควรเขียนเป็น function

### ข้อกำหนด
ให้สร้าง function ชื่อ `check_categories()` โดยมี parameter ดังนี้ 

```python 
def check_categories(df, columns): 
    ...
```

function นี้ต้องทำงานดังนี้
1. รับ DataFrame จาก df
2. รับ list ของ column ที่ต้องการตรวจจาก columns
3. ใช้ loop ผ่านแต่ละ column
4. ถ้า column นั้นมีอยู่จริงใน DataFrame ให้แสดง value_counts(dropna=False)
5. ถ้า column นั้นไม่มีอยู่ใน DataFrame ให้พิมพ์ข้อความ

>Column not found: <column name>

หลังจากสร้าง function แล้ว ให้เรียกใช้กับ column ต่อไปนี้

```python
category_columns = [ 
    "จังหวัด", 
    "เพศ", 
    "ระดับการศึกษา", 
    "อาชีพหลัก", 
    "ประเภทกลุ่มเป้าหมาย" 
]
```

In [47]:
# เขียนคำตอบของคุณใน Cell นี้

> ### เฉลยแบบฝึกหัดที่ 4
>
> ใช้ Loop ตรวจคอลัมน์ทีละชื่อ
>
> หากพบคอลัมน์ ให้ใช้
>
> ```python
> value_counts(dropna=False)
> ```
>
> เพื่อรวมค่าว่างในการนับด้วย

In [61]:
def check_categories(
    df,
    columns,
):
    for column in columns:
        print(f"\nColumn: {column}")

        if column not in df.columns:
            print(
                f"Column not found: "
                f"{column}"
            )
            continue

        category_counts = (
            df[column]
            .value_counts(
                dropna=False
            )
        )

        print(category_counts)

In [62]:
category_columns = [
    "จังหวัด",
    "เพศ",
    "ระดับการศึกษา",
    "อาชีพหลัก",
    "ประเภทกลุ่มเป้าหมาย",
]

In [63]:
check_categories(
    mso_df,
    category_columns,
)


Column: จังหวัด
จังหวัด
นครศรีธรรมราช      8
อุดรธานี           7
เชียงใหม่          5
หนองคาย            4
นครราชสีมา         4
บุรีรัมย์          4
ขอนแก่น            4
นครนายก            4
อุทัยธานี          4
สระแก้ว            3
นครพนม             3
สุราษฎร์ธานี       3
ชุมพร              3
ราชบุรี            2
ปราจีนบุรี         2
เชียงราย           2
ตาก                2
สงขลา              2
เพชรบุรี           2
มหาสารคาม          2
กาญจนบุรี          2
ชัยภูมิ            2
อุบลราชธานี        1
ประจวบคีรีขันธ์    1
ยโสธร              1
นครปฐม             1
สุพรรณบุรี         1
ระนอง              1
เพชรบูรณ์          1
นราธิวาส           1
สิงห์บุรี          1
พิษณุโลก           1
กาฬสินธุ์          1
ศรีสะเกษ           1
ลพบุรี             1
ระยอง              1
จันทบุรี           1
พังงา              1
สมุทรสงคราม        1
บึงกาฬ             1
ชลบุรี             1
มุกดาหาร           1
กระบี่             1
เลย                1
สระบุรี            1
พระนครศรีอยุธยา    1
ยะลา     

## แบบฝึกหัดที่ 5: สร้าง function สรุปคุณภาพข้อมูลเบื้องต้น
ทีมต้องการ function กลางสำหรับตรวจสอบคุณภาพข้อมูลเบื้องต้นของ DataFrame ใดก็ได้ 

ก่อนส่งต่อไปทำ data cleaning function นี้ควรสรุปข้อมูลในระดับ column 

เพื่อให้เห็นว่าแต่ละ column มีปัญหาอะไรบ้าง

### ข้อกำหนด
ให้สร้าง function ชื่อ `create_data_quality_report()` โดยมี parameter ดังนี้ 

```python 
def create_data_quality_report(df): 
    ...
```

function นี้ต้อง return DataFrame ที่มี column ต่อไปนี้
- column
- dtype
- missing_count
- missing_percent
- unique_count

หลังจากสร้าง function แล้ว ให้เรียกใช้กับ `mso_df` และเก็บผลลัพธ์ไว้ในตัวแปรชื่อ `mso_quality_report`

จากนั้นเรียงลำดับผลลัพธ์จาก `missing_percent` มากไปน้อย

In [ ]:
# เขียนคำตอบของคุณใน Cell นี้

> ### เฉลยแบบฝึกหัดที่ 5
>
> รายงานนี้รวมตัวชี้วัดระดับคอลัมน์ ได้แก่
>
> - dtype
> - จำนวนค่าว่าง
> - เปอร์เซ็นต์ค่าว่าง
> - จำนวนค่าที่ไม่ซ้ำ
>
> จากนั้นเรียงด้วย `missing_percent` จากมากไปน้อยและคืนผลลัพธ์ด้วย `return`

In [64]:
def create_data_quality_report(df):
    quality_report = pd.DataFrame(
        {
            "column": df.columns,
            "dtype": (
                df
                .dtypes
                .astype(str)
                .values
            ),
            "missing_count": (
                df
                .isna()
                .sum()
                .values
            ),
            "missing_percent": (
                df
                .isna()
                .mean()
                .mul(100)
                .round(2)
                .values
            ),
            "unique_count": (
                df
                .nunique(dropna=True)
                .values
            ),
        }
    )

    quality_report = (
        quality_report
        .sort_values(
            "missing_percent",
            ascending=False,
        )
        .reset_index(drop=True)
    )

    return quality_report

In [65]:
mso_quality_report = (
    create_data_quality_report(
        mso_df
    )
)

mso_quality_report

,column,dtype,missing_count,missing_percent,unique_count
0,รหัสประจำบ้าน,string,76,76.0,24
1,อาชีพหลัก,str,61,61.0,4
2,เพศ,str,23,23.0,2
3,ระดับการศึกษา,str,23,23.0,4
4,วันที่สร้างครัวเรือน,datetime64[us],0,0.0,100
5,รหัสครัวเรือน,string,0,0.0,100
6,วันที่แก้ไขข้อมูลล่าสุด,datetime64[us],0,0.0,100
7,อายุ,int64,0,0.0,59
8,จังหวัด,str,0,0.0,48
9,ประเภทกลุ่มเป้าหมาย,str,0,0.0,5
